# EchoFace — Wav2Lip Baseline (Colab)

This notebook walks through running a **Wav2Lip** lip-sync baseline end-to-end on Google Colab.

**Steps:**
1. Check GPU
2. Clone Wav2Lip repo
3. Install dependencies
4. Download pretrained checkpoint
5. Upload / provide a video + audio clip
6. Run inference
7. View the result
8. Save output to your project repo

> Runtime → Change runtime type → **T4 GPU** before starting.


## Step 1 — Check GPU

In [ ]:
!nvidia-smi


## Step 2 — Clone the Wav2Lip repository

In [ ]:
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip


## Step 3 — Install dependencies

In [ ]:
!pip install -q librosa==0.9.1 opencv-python-headless==4.5.5.64 numpy==1.23.5 numba==0.56.4
!pip install -q batch-face


## Step 4 — Download the pretrained checkpoint

We'll grab the `wav2lip_gan.pth` checkpoint (best visual quality). If this mirror is down,
swap in an alternate link from the Wav2Lip README.

In [ ]:
import os
os.makedirs('checkpoints', exist_ok=True)

# Primary checkpoint (Wav2Lip + GAN, higher quality lip-sync)
!wget -q -O checkpoints/wav2lip_gan.pth "https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth"

# Face detector weights used internally by Wav2Lip
!mkdir -p face_detection/detection/sfd
!wget -q -O face_detection/detection/sfd/s3fd.pth "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

print("Downloaded:")
!ls -lh checkpoints/


## Step 5 — Provide your input video + audio

Two options:

**A) Upload files directly** (small files, quick tests) — run the cell below and select a face video
(`.mp4`) and a driving audio clip (`.wav` or `.mp3`).

**B) Mount Google Drive** if your files are already there — uncomment the Drive cell instead.

In [ ]:
from google.colab import files

print("Upload your FACE VIDEO (mp4):")
video_upload = files.upload()
video_path = list(video_upload.keys())[0]

print("Upload your AUDIO (wav/mp3):")
audio_upload = files.upload()
audio_path = list(audio_upload.keys())[0]

print(f"Video: {video_path}")
print(f"Audio: {audio_path}")


In [ ]:
# Option B — Google Drive instead of manual upload (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# video_path = '/content/drive/MyDrive/EchoFace/sample_face.mp4'
# audio_path = '/content/drive/MyDrive/EchoFace/sample_audio.wav'


## Step 6 — Run Wav2Lip inference

In [ ]:
import os
os.makedirs('results', exist_ok=True)

!python inference.py \
  --checkpoint_path checkpoints/wav2lip_gan.pth \
  --face "$video_path" \
  --audio "$audio_path" \
  --outfile results/result_voice.mp4 \
  --pads 0 20 0 0


> **Common errors & fixes**
> - `No face detected` → try a video where the face is clearly visible and front-facing, or increase `--pads`.
> - `CUDA out of memory` → reduce video resolution first (see optional cell below) or restart runtime.
> - `ffmpeg` errors on audio → convert audio to `.wav` first: `!ffmpeg -i input.mp3 audio.wav`

### (Optional) Downscale video if you hit memory errors

In [ ]:
# !ffmpeg -i "$video_path" -vf scale=480:-1 resized.mp4
# video_path = "resized.mp4"


## Step 7 — View the result

In [ ]:
from IPython.display import HTML
from base64 import b64encode

mp4 = open('results/result_voice.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f"""
<video width=480 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")


## Step 8 — Save the output back to your EchoFace repo

Download the result locally, then commit it to your project (don't commit huge checkpoints —
add `checkpoints/` and large `.mp4` samples to `.gitignore` if needed).

In [ ]:
from google.colab import files
files.download('results/result_voice.mp4')


In [ ]:
# Back in your local terminal (not Colab):
#
# cd ~/Desktop/EchoFace
# mv ~/Downloads/result_voice.mp4 samples/wav2lip_baseline_result.mp4
# git add samples/wav2lip_baseline_result.mp4
# git commit -m "feat: add first Wav2Lip baseline output"
# git push origin main


---
### Next steps for EchoFace
- Try the **non-GAN** checkpoint (`wav2lip.pth`) and compare sync quality vs. visual quality.
- Swap in your own reference video + TTS-generated audio to test a full voice-cloning → lip-sync pipeline.
- Batch-process multiple clips by looping Step 6 over a folder of video/audio pairs.
- Explore face-restoration post-processing (e.g. GFPGAN) to sharpen the mouth region Wav2Lip generates.